# 🧠 Projet G02 — Fine-tuning BERT sur IMDb
## P02 : Régularisation et Généralisation

| Champ | Valeur |
|---|---|
| **Groupe** | G02 |
| **Dataset** | IMDb Reviews (D01) |
| **Modèle** | BERT-base-uncased (M02) |
| **Problématique** | P02 — Régularisation et Généralisation |
| **Méthode** | Optuna (TPE Bayésien) |
| **Métrique** | F1-score (binaire) |

> ⚠️ **Avant d'exécuter** : `Exécution > Modifier le type d'exécution > GPU T4`

## 📦 Cellule 1 — Installation des dépendances

In [ ]:
# Installation des dépendances
!pip install -q transformers datasets optuna scikit-learn seaborn tqdm
print('✅ Dépendances installées')

## ⚙️ Cellule 2 — Imports et configuration globale

In [ ]:
import os, sys, json, copy, itertools, time, warnings
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
warnings.filterwarnings('ignore')

# Répertoires de sortie
FIGURES_DIR = 'figures'
RESULTS_DIR = 'results'
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU : {torch.cuda.get_device_name(0)}')

# Hyperparamètres globaux
NUM_EPOCHS   = 3
BATCH_SIZE   = 16
WARMUP_RATIO = 0.1
MAX_LENGTH   = 200
LR_DEFAULT   = 2e-5
N_OPTUNA     = 20
SEED         = 42

plt.rcParams.update({'font.family': 'DejaVu Sans', 'axes.titlesize': 13,
                      'axes.labelsize': 11, 'figure.dpi': 130})
print('✅ Configuration prête')

## 📂 Cellule 3 — Module data_loader
> Chargement IMDb avec sous-échantillonnage équilibré adapté CPU/GPU.

In [ ]:
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader

def load_imdb_subset(num_train_per_class=300, num_val_per_class=100,
                     num_test_per_class=150, seed=SEED):
    print('Chargement du dataset IMDb...')
    raw = load_dataset('imdb')
    rng = np.random.default_rng(seed)
    def _sample(split_data, n_per_class):
        pos = [ex for ex in split_data if ex['label'] == 1]
        neg = [ex for ex in split_data if ex['label'] == 0]
        n = min(n_per_class, len(pos), len(neg))
        idx_pos = rng.choice(len(pos), n, replace=False).tolist()
        idx_neg = rng.choice(len(neg), n, replace=False).tolist()
        subset = [pos[i] for i in idx_pos] + [neg[i] for i in idx_neg]
        rng.shuffle(subset)
        return subset
    train_full = list(raw['train'])
    rng.shuffle(train_full)
    pivot = int(0.85 * len(train_full))
    raw_train, raw_val_pool = train_full[:pivot], train_full[pivot:]
    subsets = {
        'train':      _sample(raw_train,        num_train_per_class),
        'validation': _sample(raw_val_pool,      num_val_per_class),
        'test':       _sample(list(raw['test']), num_test_per_class),
    }
    for split, data in subsets.items():
        n_pos = sum(1 for d in data if d['label'] == 1)
        n_neg = sum(1 for d in data if d['label'] == 0)
        print(f'  {split:>12s}: {len(data):4d} exemples  (pos={n_pos}, neg={n_neg})')
    return subsets

class IMDbDataset(Dataset):
    def __init__(self, examples, tokenizer, max_length=256):
        self.labels = [ex['label'] for ex in examples]
        texts = [ex['text'] for ex in examples]
        self.encodings = tokenizer(texts, truncation=True, padding='max_length',
                                   max_length=max_length, return_tensors='pt')
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

def get_dataloaders(subsets, tokenizer, batch_size=16, max_length=256):
    loaders = {}
    for split, examples in subsets.items():
        ds = IMDbDataset(examples, tokenizer, max_length)
        loaders[split] = DataLoader(ds, batch_size=batch_size,
                                    shuffle=(split == 'train'), num_workers=2)
    return loaders

print('✅ Module data_loader prêt')

## 🤖 Cellule 4 — Module model_setup
> Weight decay découplé : exclu sur `bias` et `LayerNorm.weight` (pratique HuggingFace recommandée).

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW

MODEL_NAME = 'bert-base-uncased'

def get_model_and_tokenizer(model_name=MODEL_NAME, num_labels=2,
                             dropout_prob=0.1, device=DEVICE):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=num_labels,
        hidden_dropout_prob=dropout_prob,
        attention_probs_dropout_prob=dropout_prob,
        torch_dtype=torch.float32,
    )
    model = model.to(device)
    if device.type == 'cpu': torch.set_num_threads(4)
    return model, tokenizer

def build_optimizer(model, lr=2e-5, weight_decay=1e-4):
    """AdamW avec weight decay découplé (exclu sur bias et LayerNorm)."""
    no_decay = ['bias', 'LayerNorm.weight']
    grouped_params = [
        {'params': [p for n, p in model.named_parameters()
                    if not any(nd in n for nd in no_decay)],
         'weight_decay': weight_decay},
        {'params': [p for n, p in model.named_parameters()
                    if any(nd in n for nd in no_decay)],
         'weight_decay': 0.0},
    ]
    return AdamW(grouped_params, lr=lr)

print('✅ Module model_setup prêt')

## 🏋️ Cellule 5 — Module train_eval
> Early stopping (patience=2) + gradient clipping + scheduler linéaire avec warmup.

In [ ]:
import torch.nn as nn
from torch.optim.lr_scheduler import LambdaLR

def get_linear_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps):
    def lr_lambda(step):
        if step < num_warmup_steps:
            return float(step) / max(1, num_warmup_steps)
        return max(0.0, float(num_training_steps - step) /
                   max(1, num_training_steps - num_warmup_steps))
    return LambdaLR(optimizer, lr_lambda)

def train_epoch(model, loader, optimizer, scheduler, device, max_grad_norm=1.0):
    model.train()
    total_loss, total_correct, total_samples = 0.0, 0, 0
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item() * batch['labels'].size(0)
        preds = outputs.logits.argmax(dim=-1)
        total_correct += (preds == batch['labels']).sum().item()
        total_samples += batch['labels'].size(0)
    return total_loss / total_samples, total_correct / total_samples

def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels, total_loss, total_samples = [], [], 0.0, 0
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            total_loss += outputs.loss.item() * batch['labels'].size(0)
            preds = outputs.logits.argmax(dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(batch['labels'].cpu().numpy())
            total_samples += batch['labels'].size(0)
    return {'loss': total_loss / total_samples,
            'accuracy': accuracy_score(all_labels, all_preds),
            'f1': f1_score(all_labels, all_preds, average='binary'),
            'preds': all_preds, 'labels': all_labels}

def train_model(model, loaders, optimizer, num_epochs, warmup_ratio,
                device, max_grad_norm=1.0, patience=2, verbose=True):
    total_steps  = num_epochs * len(loaders['train'])
    warmup_steps = int(warmup_ratio * total_steps)
    scheduler    = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}
    best_val_loss, epochs_no_improve, best_state = float('inf'), 0, None
    t0 = time.time()
    for epoch in range(1, num_epochs + 1):
        tr_loss, tr_acc = train_epoch(model, loaders['train'], optimizer,
                                      scheduler, device, max_grad_norm)
        val_metrics = evaluate(model, loaders['validation'], device)
        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(val_metrics['loss'])
        history['val_acc'].append(val_metrics['accuracy'])
        history['val_f1'].append(val_metrics['f1'])
        if verbose:
            print(f'  Epoch {epoch}/{num_epochs} | '
                  f'train_loss={tr_loss:.4f} train_acc={tr_acc:.4f} | '
                  f'val_loss={val_metrics["loss"]:.4f} '
                  f'val_acc={val_metrics["accuracy"]:.4f} '
                  f'val_f1={val_metrics["f1"]:.4f}')
        if val_metrics['loss'] < best_val_loss:
            best_val_loss, epochs_no_improve = val_metrics['loss'], 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                if verbose: print(f'  [Early stopping] époque {epoch}')
                break
    history['train_time'] = time.time() - t0
    if best_state:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    return history

print('✅ Module train_eval prêt')

## 🏔️ Cellule 6 — Module loss_landscape
> Filter-wise normalization (Li et al., 2018) + Sharpness sur 5 directions (Keskar et al., 2017).

In [ ]:
def evaluate_on_subset(model, loader, device, n_samples=64):
    model.eval()
    total_loss, n = 0.0, 0
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            bs = batch['labels'].size(0)
            total_loss += outputs.loss.item() * bs
            n += bs
            if n >= n_samples: break
    return total_loss / max(n, 1)

def compute_loss_landscape_1d(model, loader, device, n_points=15, epsilon=0.05):
    """Perturbation 1D avec filter-wise normalization (Li et al., 2018)."""
    model.eval()
    original_params = [p.clone().detach() for p in model.parameters()]
    direction = []
    for p in model.parameters():
        d = torch.randn_like(p)
        norm_p = p.data.norm() + 1e-8
        direction.append(d / d.norm() * norm_p)
    alphas = np.linspace(-epsilon, epsilon, n_points)
    losses = []
    for alpha in alphas:
        for p, p0, d in zip(model.parameters(), original_params, direction):
            p.data = p0 + alpha * d
        losses.append(evaluate_on_subset(model, loader, device, n_samples=64))
    for p, p0 in zip(model.parameters(), original_params):
        p.data = p0.clone()
    return alphas.tolist(), losses

def compute_sharpness(model, loader, device, rho=0.05, n_directions=5):
    """Sharpness = moyenne sur n_directions de |L(θ+ε) - L(θ)|."""
    base_loss = evaluate_on_subset(model, loader, device, n_samples=128)
    original_params = [p.clone().detach() for p in model.parameters()]
    max_deltas = []
    for _ in range(n_directions):
        direction = [torch.randn_like(p) for p in model.parameters()]
        total_norm = sum(d.norm()**2 for d in direction).sqrt() + 1e-8
        direction = [rho * d / total_norm for d in direction]
        for p, p0, d in zip(model.parameters(), original_params, direction):
            p.data = p0 + d
        perturbed_loss = evaluate_on_subset(model, loader, device, n_samples=128)
        max_deltas.append(abs(perturbed_loss - base_loss))
        for p, p0 in zip(model.parameters(), original_params):
            p.data = p0.clone()
    return float(np.mean(max_deltas))

def analyze_configs(configs, loaders, device):
    results = {}
    for cfg in configs:
        label = cfg['label']
        print(f'  Analyse landscape : {label}')
        alphas, losses = compute_loss_landscape_1d(
            cfg['model'], loaders['validation'], device)
        sharpness = compute_sharpness(
            cfg['model'], loaders['validation'], device)
        best_val_f1  = max(cfg['history']['val_f1'])
        best_val_acc = max(cfg['history']['val_acc'])
        best_tr_acc  = max(cfg['history']['train_acc'])
        results[label] = {
            'alphas': alphas, 'losses': losses, 'sharpness': sharpness,
            'val_f1': best_val_f1, 'val_acc': best_val_acc,
            'train_acc': best_tr_acc,
            'generalization_gap': best_tr_acc - best_val_acc,
        }
        print(f'    Sharpness={sharpness:.5f}  Val F1={best_val_f1:.4f}  Gap={best_tr_acc - best_val_acc:.4f}')
    return results

print('✅ Module loss_landscape prêt')

## 📊 Cellule 7 — Module visualization
> 7 figures pour le rapport.

In [ ]:
def plot_training_curves(history, title='Courbes', save_name='training_curves.png'):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    epochs = range(1, len(history['train_loss']) + 1)
    axes[0].plot(epochs, history['train_loss'], 'b-o', label='Train')
    axes[0].plot(epochs, history['val_loss'],   'r-o', label='Val')
    axes[0].set_title('Loss par époque'); axes[0].set_xlabel('Époque')
    axes[0].set_ylabel('Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
    axes[1].plot(epochs, history['train_acc'], 'b-o', label='Train Acc')
    axes[1].plot(epochs, history['val_acc'],   'r-o', label='Val Acc')
    axes[1].plot(epochs, history['val_f1'],    'g--s', label='Val F1')
    axes[1].set_title('Accuracy & F1'); axes[1].set_xlabel('Époque')
    axes[1].set_ylabel('Score'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
    fig.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    path = os.path.join(FIGURES_DIR, save_name)
    fig.savefig(path, bbox_inches='tight'); plt.show(); plt.close(fig)
    return path

def plot_regularization_heatmap(results_grid, save_name='regularization_heatmap.png'):
    wds = sorted(set(k[0] for k in results_grid))
    dps = sorted(set(k[1] for k in results_grid))
    matrix = np.array([[results_grid.get((wd, dp), np.nan) for dp in dps] for wd in wds])
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(matrix, cmap='YlOrRd', aspect='auto',
                   vmin=np.nanmin(matrix)*0.97, vmax=np.nanmax(matrix))
    ax.set_xticks(range(len(dps))); ax.set_yticks(range(len(wds)))
    ax.set_xticklabels([f'{d:.2f}' for d in dps])
    ax.set_yticklabels([f'{w:.0e}' for w in wds])
    ax.set_xlabel('Dropout probability'); ax.set_ylabel('Weight decay')
    ax.set_title('Val F1 : Weight Decay x Dropout\n(Grid Search P02)')
    plt.colorbar(im, ax=ax, label='F1-score')
    for i in range(len(wds)):
        for j in range(len(dps)):
            v = matrix[i, j]
            if not np.isnan(v): ax.text(j, i, f'{v:.3f}', ha='center', va='center', fontsize=9)
    plt.tight_layout()
    path = os.path.join(FIGURES_DIR, save_name)
    fig.savefig(path, bbox_inches='tight'); plt.show(); plt.close(fig)
    return path

def plot_loss_landscape_1d(configs, save_name='loss_landscape_1d.png'):
    fig, ax = plt.subplots(figsize=(9, 5))
    colors = plt.cm.tab10(np.linspace(0, 1, len(configs)))
    for cfg, color in zip(configs, colors):
        alphas = np.array(cfg['alphas']); losses = np.array(cfg['losses'])
        ax.plot(alphas, losses, label=cfg['label'], color=color, linewidth=2)
        idx_min = np.argmin(losses)
        ax.scatter(alphas[idx_min], losses[idx_min], color=color, s=80, zorder=5)
    ax.set_xlabel('Direction de perturbation α'); ax.set_ylabel('Loss')
    ax.set_title('Loss Landscape 1D — Filter-wise normalization (Li et al., 2018)')
    ax.legend(loc='upper right'); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    path = os.path.join(FIGURES_DIR, save_name)
    fig.savefig(path, bbox_inches='tight'); plt.show(); plt.close(fig)
    return path

def plot_sharpness_comparison(sharpness_dict, save_name='sharpness_comparison.png'):
    labels = list(sharpness_dict.keys()); values = list(sharpness_dict.values())
    colors = ['#2196F3' if v == min(values) else '#FF5722' for v in values]
    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.bar(labels, values, color=colors, edgecolor='black', linewidth=0.5)
    for bar, v in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + max(values)*0.01,
                f'{v:.4f}', ha='center', va='bottom', fontsize=9)
    ax.set_xlabel('Configuration'); ax.set_ylabel('Sharpness')
    ax.set_title('Sharpness des minima (Bleu=plus plat=meilleure généralisation)')
    ax.grid(True, axis='y', alpha=0.3)
    plt.xticks(rotation=25, ha='right'); plt.tight_layout()
    path = os.path.join(FIGURES_DIR, save_name)
    fig.savefig(path, bbox_inches='tight'); plt.show(); plt.close(fig)
    return path

def plot_optuna_history(study, save_name='optuna_history.png'):
    import optuna
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    trial_numbers = [t.number for t in study.trials if t.value is not None]
    values = [t.value for t in study.trials if t.value is not None]
    best_so_far = np.maximum.accumulate(values)
    axes[0].plot(trial_numbers, values, 'o', color='steelblue', alpha=0.6, label='Trial')
    axes[0].plot(trial_numbers, best_so_far, '-', color='crimson', linewidth=2.5, label='Best')
    axes[0].set_xlabel('Trial'); axes[0].set_ylabel('Val F1-score')
    axes[0].set_title('Convergence Optuna TPE'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
    try:
        importance = optuna.importance.get_param_importances(study)
        axes[1].barh(list(importance.keys()), list(importance.values()),
                     color='teal', edgecolor='black')
        axes[1].set_title('Importance des hyperparamètres')
    except: axes[1].text(0.5, 0.5, 'Non disponible', ha='center', va='center',
                          transform=axes[1].transAxes)
    fig.suptitle('Optimisation Bayésienne — Optuna G02 P02', fontsize=14, fontweight='bold')
    plt.tight_layout()
    path = os.path.join(FIGURES_DIR, save_name)
    fig.savefig(path, bbox_inches='tight'); plt.show(); plt.close(fig)
    return path

def plot_confusion_matrix_fig(y_true, y_pred,
                               class_names=('Négatif', 'Positif'),
                               save_name='confusion_matrix.png'):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_xlabel('Prédit'); ax.set_ylabel('Réel')
    ax.set_title('Matrice de Confusion — Test set'); plt.tight_layout()
    path = os.path.join(FIGURES_DIR, save_name)
    fig.savefig(path, bbox_inches='tight'); plt.show(); plt.close(fig)
    return path

def plot_generalization_gap(configs_gap, save_name='generalization_gap.png'):
    labels = list(configs_gap.keys())
    train_accs = [configs_gap[l]['train_acc'] for l in labels]
    val_accs   = [configs_gap[l]['val_acc']   for l in labels]
    gaps = [tr - va for tr, va in zip(train_accs, val_accs)]
    x = np.arange(len(labels)); width = 0.3
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x - width/2, train_accs, width, label='Train Acc', color='#42A5F5', edgecolor='black')
    ax.bar(x + width/2, val_accs,   width, label='Val Acc',   color='#EF5350', edgecolor='black')
    ax2 = ax.twinx()
    ax2.plot(x, gaps, 'ko--', linewidth=1.5, markersize=7, label='Gap')
    ax2.set_ylabel('Généralisation gap')
    ax.set_xticks(x); ax.set_xticklabels(labels, rotation=25, ha='right')
    ax.set_ylabel('Accuracy')
    ax.set_title('Écart train/val par configuration de régularisation')
    ax.legend(loc='upper left'); ax2.legend(loc='upper right')
    ax.grid(True, axis='y', alpha=0.3); plt.tight_layout()
    path = os.path.join(FIGURES_DIR, save_name)
    fig.savefig(path, bbox_inches='tight'); plt.show(); plt.close(fig)
    return path

print('✅ Module visualization prêt')

## ▶️ ÉTAPE 1 — Chargement des données IMDb

In [ ]:
subsets = load_imdb_subset(
    num_train_per_class=300,
    num_val_per_class=100,
    num_test_per_class=150,
    seed=SEED,
)
print('✅ Données prêtes')

## ▶️ ÉTAPE 2 — Grid Search P02

**Protocole imposé par le sujet P02 :**
- `weight_decay` ∈ {1e-5, 1e-4, 1e-3, 1e-2}
- `dropout_prob` ∈ {0.0, 0.1, 0.3}
- 4 × 3 = **12 combinaisons**


In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

WD_GRID      = [1e-5, 1e-4, 1e-3, 1e-2]
DROPOUT_GRID = [0.0, 0.1, 0.3]

grid_results = {}
total = len(WD_GRID) * len(DROPOUT_GRID)
print(f'Grid Search P02 — {total} combinaisons sur {DEVICE}\n')

for i, (wd, dp) in enumerate(itertools.product(WD_GRID, DROPOUT_GRID), 1):
    label = f'WD={wd:.0e} Drop={dp:.1f}'
    print(f'[{i:2d}/{total}] {label}')
    model, tokenizer = get_model_and_tokenizer(dropout_prob=dp, device=DEVICE)
    loaders   = get_dataloaders(subsets, tokenizer, batch_size=BATCH_SIZE, max_length=MAX_LENGTH)
    optimizer = build_optimizer(model, lr=LR_DEFAULT, weight_decay=wd)
    history   = train_model(model, loaders, optimizer, num_epochs=NUM_EPOCHS,
                             warmup_ratio=WARMUP_RATIO, device=DEVICE, patience=2, verbose=True)
    best_val_f1  = max(history['val_f1'])
    best_val_acc = max(history['val_acc'])
    best_tr_acc  = max(history['train_acc'])
    grid_results[(wd, dp)] = {
        'weight_decay': wd, 'dropout_prob': dp,
        'val_f1': best_val_f1, 'val_acc': best_val_acc,
        'train_acc': best_tr_acc,
        'gen_gap': best_tr_acc - best_val_acc,
        'history': history, 'model': model,
    }

best_grid_key  = max(grid_results, key=lambda k: grid_results[k]['val_f1'])
worst_grid_key = min(grid_results, key=lambda k: grid_results[k]['val_f1'])
print(f'\nGrid Best  : WD={best_grid_key[0]:.0e} Drop={best_grid_key[1]} → F1={grid_results[best_grid_key]["val_f1"]:.4f}')
print(f'Grid Worst : WD={worst_grid_key[0]:.0e} Drop={worst_grid_key[1]} → F1={grid_results[worst_grid_key]["val_f1"]:.4f}')
plot_regularization_heatmap({k: v['val_f1'] for k, v in grid_results.items()})
print('✅ Grid Search terminé')

## ▶️ ÉTAPE 3 — Optimisation Optuna TPE

> Méthode assignée au groupe G02 : Optuna (TPE Bayésien).

In [ ]:
from optuna.samplers import TPESampler
from optuna.pruners  import MedianPruner

_SHARED_DATA_OPTUNA = {'subsets': subsets}

SEARCH_SPACE = {
    'weight_decay': ('log_float', 1e-5, 1e-2),
    'dropout_prob': ('float',     0.0,  0.3),
    'lr':           ('log_float', 1e-6, 5e-4),
    'batch_size':   ('categorical', [8, 16]),
    'warmup_ratio': ('float',     0.0,  0.15),
    'num_epochs':   ('int',       2,    4),
}

def suggest_params(trial):
    params = {}
    for name, spec in SEARCH_SPACE.items():
        kind = spec[0]
        if kind == 'log_float': params[name] = trial.suggest_float(name, spec[1], spec[2], log=True)
        elif kind == 'float':   params[name] = trial.suggest_float(name, spec[1], spec[2])
        elif kind == 'categorical': params[name] = trial.suggest_categorical(name, spec[1])
        elif kind == 'int':     params[name] = trial.suggest_int(name, spec[1], spec[2])
    return params

def objective(trial):
    params = suggest_params(trial)
    model, tokenizer = get_model_and_tokenizer(
        dropout_prob=params['dropout_prob'], device=DEVICE)
    loaders = get_dataloaders(_SHARED_DATA_OPTUNA['subsets'], tokenizer,
                               batch_size=params['batch_size'], max_length=MAX_LENGTH)
    optimizer = build_optimizer(model, lr=params['lr'],
                                 weight_decay=params['weight_decay'])
    history = train_model(model, loaders, optimizer,
                           num_epochs=params['num_epochs'],
                           warmup_ratio=params['warmup_ratio'],
                           device=DEVICE, patience=2, verbose=False)
    val_f1 = max(history['val_f1'])
    trial.report(val_f1, step=len(history['val_f1']))
    if trial.should_prune(): raise optuna.TrialPruned()
    return val_f1

sampler = TPESampler(seed=SEED, n_startup_trials=5)
pruner  = MedianPruner(n_startup_trials=5, n_warmup_steps=1)
study   = optuna.create_study(direction='maximize', sampler=sampler, pruner=pruner)
print(f'Optuna TPE — {N_OPTUNA} trials sur {DEVICE}\n')
study.optimize(objective, n_trials=N_OPTUNA, show_progress_bar=True, catch=(Exception,))
best_params = study.best_params
print(f'\nMeilleur Val F1 : {study.best_value:.4f}')
print(f'Meilleurs HP    : {best_params}')
plot_optuna_history(study)
print('✅ Optuna terminé')

## ▶️ ÉTAPE 4 — Entraînement des 5 configurations de référence

In [ ]:
CONFIGS_TO_COMPARE = {
    'Défaut (WD=0, Drop=0.1)': {'weight_decay': 0.0, 'dropout_prob': 0.1, 'lr': LR_DEFAULT},
    f'Grid Best (WD={best_grid_key[0]:.0e}, Drop={best_grid_key[1]:.1f})': {
        'weight_decay': best_grid_key[0], 'dropout_prob': best_grid_key[1], 'lr': LR_DEFAULT},
    f'Grid Worst (WD={worst_grid_key[0]:.0e}, Drop={worst_grid_key[1]:.1f})': {
        'weight_decay': worst_grid_key[0], 'dropout_prob': worst_grid_key[1], 'lr': LR_DEFAULT},
    'Fort WD (WD=1e-2, Drop=0)': {'weight_decay': 1e-2, 'dropout_prob': 0.0, 'lr': LR_DEFAULT},
    'Optuna Best': {
        'weight_decay': best_params.get('weight_decay', 1e-4),
        'dropout_prob': best_params.get('dropout_prob', 0.1),
        'lr':           best_params.get('lr', LR_DEFAULT),
    },
}

trained_configs  = []
tokenizer_shared = None

for cfg_name, hp in CONFIGS_TO_COMPARE.items():
    print(f'\n── Configuration : {cfg_name} ──')
    model, tokenizer = get_model_and_tokenizer(
        dropout_prob=hp['dropout_prob'], device=DEVICE)
    if tokenizer_shared is None: tokenizer_shared = tokenizer
    loaders   = get_dataloaders(subsets, tokenizer, batch_size=BATCH_SIZE, max_length=MAX_LENGTH)
    optimizer = build_optimizer(model, lr=hp['lr'], weight_decay=hp['weight_decay'])
    history   = train_model(model, loaders, optimizer,
                             num_epochs=NUM_EPOCHS, warmup_ratio=WARMUP_RATIO,
                             device=DEVICE, patience=2, verbose=True)
    trained_configs.append({'label': cfg_name, 'model': model, 'history': history, 'hp': hp})
    safe = cfg_name.replace(' ', '_').replace('/', '_').replace('=', '').replace(',', '')
    plot_training_curves(history, title=f'Entraînement — {cfg_name}',
                         save_name=f'training_{safe}.png')

print('\n✅ 5 configurations entraînées')

## ▶️ ÉTAPE 5 — Analyse Loss Landscape & Sharpness

In [ ]:
print('Analyse Loss Landscape (peut prendre quelques minutes)...')
loaders_shared = get_dataloaders(subsets, tokenizer_shared,
                                  batch_size=BATCH_SIZE, max_length=MAX_LENGTH)
landscape_results = analyze_configs(trained_configs, loaders_shared, DEVICE)
print('\n✅ Landscape analysé')

## ▶️ ÉTAPE 6 — Génération des figures de synthèse

In [ ]:
# Loss Landscape 1D
landscape_plot_data = [
    {'label': label, 'alphas': np.array(res['alphas']), 'losses': np.array(res['losses'])}
    for label, res in landscape_results.items()
]
plot_loss_landscape_1d(landscape_plot_data)

# Sharpness
sharpness_dict = {label: res['sharpness'] for label, res in landscape_results.items()}
plot_sharpness_comparison(sharpness_dict)

# Généralisation gap
gap_dict = {
    label: {'train_acc': res['train_acc'], 'val_acc': res['val_acc']}
    for label, res in landscape_results.items()
}
plot_generalization_gap(gap_dict)
print('✅ Figures générées')

## ▶️ ÉTAPE 7 — Évaluation finale sur le Test Set

In [ ]:
# Sélectionner la meilleure config
best_cfg = max(trained_configs, key=lambda c: max(c['history']['val_f1']))
print(f'Meilleure configuration (val F1) : {best_cfg["label"]}')

# Évaluation sur test set
test_results = evaluate(best_cfg['model'], loaders_shared['test'], DEVICE)
print(f'\n── Résultats Test Set ──')
print(f'Accuracy  : {test_results["accuracy"]:.4f}')
print(f'F1-score  : {test_results["f1"]:.4f}')
print(f'Loss      : {test_results["loss"]:.4f}')

# Matrice de confusion
plot_confusion_matrix_fig(test_results['labels'], test_results['preds'])

# Résumé complet
print('\n── RÉSUMÉ COMPLET ──')
print(f'{'Config':30s} | {'Val F1':>8s} | {'Sharpness':>10s} | {'Gap':>8s}')
print('-' * 65)
for label, res in landscape_results.items():
    marker = ' ← BEST' if label == best_cfg['label'] else ''
    print(f'{label:30s} | {res["val_f1"]:>8.4f} | {res["sharpness"]:>10.5f} | {res["generalization_gap"]:>8.4f}{marker}')

# Sauvegarde JSON
final_summary = {
    'groupe': 'G02', 'dataset': 'IMDb (D01)', 'modele': 'BERT-base-uncased (M02)',
    'problematique': 'P02 — Régularisation et Généralisation', 'methode': 'Optuna TPE',
    'best_config': best_cfg['label'], 'best_hp': best_cfg['hp'],
    'test_metrics': {k: v for k, v in test_results.items() if k not in ('preds', 'labels')},
    'sharpness': sharpness_dict,
    'generalization_gaps': {k: v['train_acc'] - v['val_acc'] for k, v in gap_dict.items()},
    'optuna_best_params': best_params, 'optuna_best_val_f1': study.best_value,
    'grid_best_key': list(best_grid_key),
    'grid_best_val_f1': grid_results[best_grid_key]['val_f1'],
}
with open(os.path.join(RESULTS_DIR, 'final_summary.json'), 'w') as f:
    json.dump(final_summary, f, indent=2, default=float)
print(f'\n✅ Résultats sauvegardés dans {RESULTS_DIR}/final_summary.json')

## 💾 Cellule 15 — Sauvegarde sur Google Drive (optionnel)

In [ ]:
import shutil
from google.colab import drive

# Monter Google Drive
drive.mount('/content/drive')

# Créer le dossier de destination
drive_path = '/content/drive/My Drive/G02_BERT_IMDb'
os.makedirs(drive_path, exist_ok=True)

# Copier figures et résultats
shutil.copytree(FIGURES_DIR, os.path.join(drive_path, 'figures'), dirs_exist_ok=True)
shutil.copytree(RESULTS_DIR, os.path.join(drive_path, 'results'), dirs_exist_ok=True)
print(f'✅ Fichiers copiés dans Google Drive : {drive_path}')